In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read bronze accounts table
bronze_accounts = spark.table("digital_banking.bronze.bronze_accounts")

# Read reference tables
silver_customers = spark.table("digital_banking.silver.silver_customers")
silver_branches = spark.table("digital_banking.silver.silver_branches")

print(f"Total records in bronze_accounts: {bronze_accounts.count()}")
print(f"Total customers in silver_customers: {silver_customers.count()}")
print(f"Total branches in silver_branches: {silver_branches.count()}")

In [0]:

# Check for duplicate account_ids
duplicates = bronze_accounts.groupBy("account_id").count().filter("count > 1")
duplicates_count = duplicates.count()

print(f"Duplicate account_ids found: {duplicates_count}")

if duplicates_count > 0:
    print("\nSample duplicate account_ids:")
    display(duplicates.limit(10))
    
    # Add row number to identify duplicates and keep only the first occurrence
    window_spec = Window.partitionBy("account_id").orderBy(F.col("updated_at").desc())
    bronze_accounts_dedup = bronze_accounts.withColumn("row_num", F.row_number().over(window_spec)) \
                                           .filter("row_num = 1") \
                                           .drop("row_num")
    print(f"\nAfter deduplication: {bronze_accounts_dedup.count()} records")
else:
    bronze_accounts_dedup = bronze_accounts
    print("All account_ids are unique")

In [0]:
# Create sets of valid customer_ids and branch_ids for validation
valid_customer_ids = silver_customers.select("customer_id").distinct()
valid_branch_ids = silver_branches.select("branch_id").distinct()

# Add flags for referential integrity using marker columns
# Strategy: Add a marker column (value=1) to reference tables before joining
# After left join: matched records have marker=1, unmatched have marker=NULL
# Coalesce converts NULL to 0 for a clean 0/1 flag
accounts_with_flags = bronze_accounts_dedup.alias("a") \
    .join(valid_customer_ids.withColumn("customer_exists", F.lit(1)), "customer_id", "left") \
    .join(valid_branch_ids.withColumn("branch_exists", F.lit(1)), "branch_id", "left") \
    .select(
        "a.*",
        F.coalesce(F.col("customer_exists"), F.lit(0)).alias("customer_exists"),
        F.coalesce(F.col("branch_exists"), F.lit(0)).alias("branch_exists")
    )

# Count invalid references
invalid_customer_refs = accounts_with_flags.filter("customer_exists = 0").count()
invalid_branch_refs = accounts_with_flags.filter("branch_exists = 0").count()
both_invalid = accounts_with_flags.filter("customer_exists = 0 AND branch_exists = 0").count()
valid_both = accounts_with_flags.filter("customer_exists = 1 AND branch_exists = 1").count()

print(f"Accounts with invalid customer_id: {invalid_customer_refs}")
print(f"Accounts with invalid branch_id: {invalid_branch_refs}")
print(f"Accounts with both invalid: {both_invalid}")
print(f"Accounts with valid customer AND branch: {valid_both}")

In [0]:
# Define valid values
valid_account_statuses = ['Active', 'Inactive', 'Closed', 'Dormant', 'Suspended']
valid_account_types = ['Savings', 'Checking', 'Business', 'Credit']

# First, let's check what values actually exist in the data
print("\n=== Current Data Values ===")
print("\nAccount Status values:")
display(bronze_accounts_dedup.groupBy("account_status").count().orderBy("count", ascending=False))

print("\nAccount Type values:")
display(bronze_accounts_dedup.groupBy("account_type").count().orderBy("count", ascending=False))

# Add validation flags
accounts_validated = accounts_with_flags \
    .withColumn(
        "is_valid_status",
        F.when(F.col("account_status").isin(valid_account_statuses), 1).otherwise(0)
    ) \
    .withColumn(
        "is_valid_type",
        F.when(F.col("account_type").isin(valid_account_types), 1).otherwise(0)
    ) \
    .withColumn(
        "is_valid_opening_date",
        F.when(
            (F.col("opening_date").isNotNull()) & 
            (F.to_date(F.col("opening_date")) <= F.current_date()),
            1
        ).otherwise(0)
    ) \
    .withColumn(
        "is_valid_closing_date",
        F.when(
            (F.col("closing_date").isNull()) | 
            (
                (F.to_date(F.col("closing_date")) >= F.to_date(F.col("opening_date"))) &
                (F.to_date(F.col("closing_date")) <= F.current_date())
            ),
            1
        ).otherwise(0)
    )

# Summary 
print(f"Invalid account status: {accounts_validated.filter('is_valid_status = 0').count()}")
print(f"Invalid account type: {accounts_validated.filter('is_valid_type = 0').count()}")
print(f"Invalid opening date: {accounts_validated.filter('is_valid_opening_date = 0').count()}")
print(f"Invalid closing date: {accounts_validated.filter('is_valid_closing_date = 0').count()}")

In [0]:
# Keep only accounts that:
# 1. Have valid customer references (customer exists in silver_customers)
# 2. Have valid branch references (branch exists in silver_branches)
# 3. Have valid status, type, and dates

silver_accounts = accounts_validated.filter(
    """
    customer_exists = 1 AND 
    branch_exists = 1 AND
    is_valid_status = 1 AND
    is_valid_type = 1 AND
    is_valid_opening_date = 1 AND
    is_valid_closing_date = 1
    """
)

# Cast date columns to proper date type
silver_accounts = silver_accounts \
    .withColumn("opening_date", F.to_date(F.col("opening_date"))) \
    .withColumn("closing_date", F.to_date(F.col("closing_date"))) \
    .withColumn("updated_at", F.to_date(F.col("updated_at"))) \
    .withColumn("interest_rate", F.col("interest_rate").cast("decimal(5,2)"))

# Add metadata columns
silver_accounts = silver_accounts \
    .withColumn("processed_at", F.current_timestamp()) \
    .withColumn("data_layer", F.lit("silver"))

# Select final columns (exclude validation flags)
final_columns = [
    "account_id",
    "customer_id",
    "branch_id",
    "account_type",
    "account_status",
    "opening_date",
    "closing_date",
    "currency",
    "account_tier",
    "interest_rate",
    "updated_at",
    "processed_at",
    "data_layer"
]

silver_accounts_final = silver_accounts.select(final_columns)

print("\n=== Final Statistics ===")
print(f"Total bronze records: {bronze_accounts.count()}")
print(f"After deduplication: {bronze_accounts_dedup.count()}")
print(f"Valid silver records: {silver_accounts_final.count()}")
print(f"Records filtered out: {bronze_accounts_dedup.count() - silver_accounts_final.count()}")
print(f"Data quality pass rate: {(silver_accounts_final.count() / bronze_accounts_dedup.count() * 100):.2f}%")
display(silver_accounts_final.limit(10))

In [0]:
# CREATE REJECTED RECORDS TABLE

# Identify all rejected records (records that did NOT pass all validations)
rejected_accounts = accounts_validated.filter(
    """
    customer_exists = 0 OR 
    branch_exists = 0 OR
    is_valid_status = 0 OR
    is_valid_type = 0 OR
    is_valid_opening_date = 0 OR
    is_valid_closing_date = 0
    """
)

# Add detailed rejection reasons
rejected_accounts_with_reasons = rejected_accounts \
    .withColumn(
        "rejection_reasons",
        F.concat_ws(
            "; ",
            F.when(F.col("customer_exists") == 0, "Invalid customer reference").otherwise(None),
            F.when(F.col("branch_exists") == 0, "Invalid branch reference").otherwise(None),
            F.when(F.col("is_valid_status") == 0, "Invalid account status").otherwise(None),
            F.when(F.col("is_valid_type") == 0, "Invalid account type").otherwise(None),
            F.when(F.col("is_valid_opening_date") == 0, "Invalid opening date").otherwise(None),
            F.when(F.col("is_valid_closing_date") == 0, "Invalid closing date").otherwise(None)
        )
    )

# Add metadata columns
rejected_accounts_with_reasons = rejected_accounts_with_reasons \
    .withColumn("rejected_at", F.current_timestamp()) \
    .withColumn("data_layer", F.lit("rejected"))

# Select final columns for rejected table
rejected_columns = [
    "account_id",
    "customer_id",
    "branch_id",
    "account_type",
    "account_status",
    "opening_date",
    "closing_date",
    "currency",
    "account_tier",
    "interest_rate",
    "updated_at",
    "customer_exists",
    "branch_exists",
    "is_valid_status",
    "is_valid_type",
    "is_valid_opening_date",
    "is_valid_closing_date",
    "rejection_reasons",
    "rejected_at",
    "data_layer"
]

rejected_accounts_final = rejected_accounts_with_reasons.select(rejected_columns)

print(f"  - Invalid customer reference: {rejected_accounts.filter('customer_exists = 0').count()}")
print(f"  - Invalid branch reference: {rejected_accounts.filter('branch_exists = 0').count()}")
print(f"  - Invalid account status: {rejected_accounts.filter('is_valid_status = 0').count()}")
print(f"  - Invalid account type: {rejected_accounts.filter('is_valid_type = 0').count()}")
print(f"  - Invalid opening date: {rejected_accounts.filter('is_valid_opening_date = 0').count()}")
print(f"  - Invalid closing date: {rejected_accounts.filter('is_valid_closing_date = 0').count()}")

display(rejected_accounts_final.select("account_id", "customer_id", "branch_id", "account_status", "account_type", "rejection_reasons").limit(10))

In [0]:
# Write rejected records to Delta table with overwrite mode
rejected_accounts_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("digital_banking.silver.silver_accounts_rejected")

print(f"\nRejected record count: {spark.table('digital_banking.silver.silver_accounts_rejected').count()}")


In [0]:
# ====================================
# WRITE TO SILVER TABLE
# ====================================

# Write to Delta table with overwrite mode
silver_accounts_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("digital_banking.silver.silver_accounts")

# Show sample of the silver table
display(spark.table("digital_banking.silver.silver_accounts").limit(10))